Importing the required libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error,mean_absolute_percentage_error,r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
import warnings
warnings.filterwarnings('ignore')

Importing the dataset

In [3]:
df = pd.read_csv("Air Traffic Data Final.csv")
df.head()

,Date,dom_pass,int_pass,dom_freight,int_freight,gdp,population,jet_fuel,inflation,unemployment,exchange_rate
0,2009-01-01,3288004,885435,20832,11675,1.341888e+12,1225524753,71.75,10.88,7.66,48.70
1,2009-01-02,3293220,757168,18645,12482,1.341888e+12,1225524753,61.97,10.88,7.66,49.25
2,2009-01-03,3122400,848046,23046,15359,1.341888e+12,1225524753,65.01,10.88,7.66,51.13
3,2009-01-04,3266686,861715,21623,14512,1.341888e+12,1225524753,68.55,10.88,7.66,49.97
4,2009-01-05,3883887,898410,19534,14586,1.341888e+12,1225524753,72.22,10.88,7.66,48.51


Checking for null values

In [4]:
df.isna().sum()

Date             0
dom_pass         0
int_pass         0
dom_freight      0
int_freight      0
gdp              0
population       0
jet_fuel         0
inflation        0
unemployment     0
exchange_rate    0
dtype: int64

In [5]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

Defining the target features

In [9]:
target_cols = [
    'dom_pass',
    'dom_freight',
]

In [10]:
feature_cols = [col for col in df.columns if col not in ['Date'] + target_cols]

Lag generation for time-dependency

In [12]:
def create_lag_features(df, cols, lags=12):
    for col in cols:
        for i in range(1, lags + 1):
            df[f'{col}_lag{i}'] = df[col].shift(i)
    df = df.dropna().reset_index(drop=True)
    return df

df = create_lag_features(df, target_cols, lags=3)
df.shape

(186, 17)

Model Evaluation function

In [13]:
def evaluate_model(y_true, y_pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true,y_pred),
        'MSE': mean_squared_error(y_true,y_pred)
    }

Train Test Split the data

In [14]:
X = df[feature_cols + [f'{t}_lag{i}' for t in target_cols for i in range(1, 4)]]
results = []

# Linear Regression

In [15]:
for target in target_cols:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    metrics = evaluate_model(y_test, preds)
    print(f"\n🔹 Target: {target}")
    print("Model: Linear Regression")
    print(metrics)



🔹 Target: dom_pass
Model: Linear Regression
{'RMSE': 2566715.7766295485, 'MAE': 2078025.2760165154, 'R2': -1.2938371747857724, 'MAPE': 0.1957558702248147, 'MSE': 6588029877999.027}

🔹 Target: dom_freight
Model: Linear Regression
{'RMSE': 16960.469443017468, 'MAE': 15399.648336241997, 'R2': -9.360857112035067, 'MAPE': 0.25504528025376033, 'MSE': 287657523.7275293}


# Random Forest Regressor

In [18]:
for target in target_cols:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RandomForestRegressor(n_estimators=500, random_state=42)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    metrics = evaluate_model(y_test, preds)
    print(f"\n🔹 Target: {target}")
    print("Model: Random Forest")
    print(metrics)



🔹 Target: dom_pass
Model: Random Forest
{'RMSE': 1449471.8754317954, 'MAE': 987667.4297894738, 'R2': 0.2684793121927872, 'MAPE': 0.09266023003254288, 'MSE': 2100968717667.7664}

🔹 Target: dom_freight
Model: Random Forest
{'RMSE': 4855.069930157936, 'MAE': 3949.9965263157906, 'R2': 0.1509936742019784, 'MAPE': 0.06687517284004837, 'MSE': 23571704.02672379}


# XGBoost Regressor

In [17]:
for target in target_cols:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    metrics = evaluate_model(y_test, preds)
    print(f"\n🔹 Target: {target}")
    print("Model: XGBoost")
    print(metrics)



🔹 Target: dom_pass
Model: XGBoost
{'RMSE': 1919924.153965184, 'MAE': 1578938.644736842, 'R2': -0.283439040184021, 'MAPE': 0.13667046662681198, 'MSE': 3686108756978.9277}

🔹 Target: dom_freight
Model: XGBoost
{'RMSE': 6202.035877179985, 'MAE': 4955.117907072368, 'R2': -0.38544225692749023, 'MAPE': 0.08222929655738552, 'MSE': 38465249.0218277}


# LightGBM Regressor

In [18]:
from lightgbm import LGBMRegressor

for target in target_cols:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    metrics = evaluate_model(y_test, preds)
    print(f"\n🔹 Target: {target}")
    print("Model: LightGBM")
    print(metrics)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000201 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 714
[LightGBM] [Info] Number of data points in the train set: 151, number of used features: 16
[LightGBM] [Info] Start training from score 6891577.794702
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

# SVR

In [19]:
from sklearn.svm import SVR

for target in target_cols:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = SVR(kernel='rbf', C=100, gamma=0.1)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    metrics = evaluate_model(y_test, preds)
    print(f"\n🔹 Target: {target}")
    print("Model: SVR")
    print(metrics)




🔹 Target: domestic passengers
Model: SVR
{'RMSE': 6499893.555839054, 'MAE': 6275070.223277221, 'R2': -13.710231784448016, 'MAPE': 0.5075556519024955, 'MSE': 42248616237238.06}

🔹 Target: international passengers
Model: SVR
{'RMSE': 1153319.9239684942, 'MAE': 1074622.0133871834, 'R2': -3.254445617351089, 'MAPE': 0.6901179794400107, 'MSE': 1330146847022.6934}

🔹 Target: domestic freight(in tonne)
Model: SVR
{'RMSE': 20666.52274902695, 'MAE': 19975.66863829954, 'R2': -14.38348624957246, 'MAPE': 0.3218238586012612, 'MSE': 427105162.5360484}

🔹 Target: international freight(in tonne)
Model: SVR
{'RMSE': 4875.49570827917, 'MAE': 3856.4299903492097, 'R2': -0.17623311355076465, 'MAPE': 0.7473173588364898, 'MSE': 23770458.4014486}
